# Crime Scene Six vision server (Kaggle)
Settings: **Accelerator = GPU T4 x2 (or P100)**, **Internet = On**. Add Kaggle Secrets: `NGROK_TOKEN`, `KAGGLE_API_KEY` (any long random string you also put in your PC's .env), optional `NGROK_DOMAIN` (your free static ngrok domain, recommended so the URL never changes). Then **Run All**. Copy the printed PUBLIC URL into `KAGGLE_OLLAMA_URL` in your .env.

This notebook is the only AI backend. It checks the GPU, starts Ollama, runs a real image test, streams replies through the tunnel (so long answers never hit a timeout), cancels abandoned requests (so the queue never jams), and restarts anything that dies.

In [ ]:
import shutil, subprocess
if not shutil.which('nvidia-smi') or subprocess.run(['nvidia-smi', '-L'], capture_output=True).returncode != 0:
    raise RuntimeError('NO GPU in this session. In Kaggle: right panel > Session options > Accelerator > GPU T4 x2 (or P100), then Run All again. Without a GPU the model is far too slow to use.')
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import os, subprocess, time, requests as rq
os.environ.update({'OLLAMA_HOST': '127.0.0.1:11434', 'OLLAMA_KEEP_ALIVE': '24h', 'OLLAMA_NUM_PARALLEL': '1',
                   'OLLAMA_MAX_LOADED_MODELS': '1', 'OLLAMA_FLASH_ATTENTION': '1', 'OLLAMA_CONTEXT_LENGTH': '8192'})
OLLAMA = 'http://127.0.0.1:11434'
MODEL = 'qwen2.5vl:7b'   # change here and in KAGGLE_MODEL on your PC to use another vision model

def start_ollama():
    subprocess.run(['pkill', '-f', 'ollama serve'], check=False); time.sleep(2)
    subprocess.Popen(['ollama', 'serve'], stdout=open('/tmp/ollama.log', 'a'), stderr=subprocess.STDOUT)
    for _ in range(90):
        try:
            if rq.get(OLLAMA + '/api/tags', timeout=3).ok:
                return
        except Exception:
            pass
        time.sleep(1)
    raise RuntimeError('ollama did not start:\n' + open('/tmp/ollama.log').read()[-1500:])

start_ollama()
for attempt in range(4):
    if subprocess.run(['ollama', 'pull', MODEL]).returncode == 0:
        break
    print('pull failed, retrying...'); time.sleep(10)
else:
    raise RuntimeError('could not pull ' + MODEL)


In [ ]:
!pip -q install -U flask pyngrok requests pillow
import json, os, threading
from kaggle_secrets import UserSecretsClient
from flask import Flask, request, Response, stream_with_context
sec = UserSecretsClient()
NGROK = sec.get_secret('NGROK_TOKEN'); API_KEY = sec.get_secret('KAGGLE_API_KEY')
try:
    DOMAIN = sec.get_secret('NGROK_DOMAIN')
except Exception:
    DOMAIN = None

app = Flask(__name__)

def authed():
    return request.headers.get('Authorization') == f'Bearer {API_KEY}'

@app.route('/health')
def health():
    if not authed():
        return Response('unauthorized', 401)
    try:
        ps = rq.get(OLLAMA + '/api/ps', timeout=5).json()
        return Response(json.dumps({'ok': True, 'loaded': [m.get('name') for m in ps.get('models', [])]}), content_type='application/json')
    except Exception as e:
        return Response(json.dumps({'ok': False, 'error': str(e)[:200]}), 503, content_type='application/json')

@app.route('/', defaults={'p': ''}, methods=['GET', 'POST'])
@app.route('/<path:p>', methods=['GET', 'POST'])
def proxy(p):
    if not authed():
        return Response('unauthorized', 401)
    try:
        up = rq.request(request.method, f'{OLLAMA}/{p}', data=request.get_data(), headers={'Content-Type': 'application/json'}, stream=True, timeout=(10, 900))
    except Exception as e:
        return Response(json.dumps({'error': 'ollama unreachable: ' + str(e)[:200]}), 502, content_type='application/json')
    def gen():
        try:
            for chunk in up.iter_content(chunk_size=None):
                if chunk:
                    yield chunk
        finally:
            up.close()   # client gone (timeout/Ctrl+C) -> upstream closed -> Ollama stops generating, queue never jams
    return Response(stream_with_context(gen()), up.status_code, content_type=up.headers.get('Content-Type', 'application/json'))

threading.Thread(target=lambda: app.run(host='127.0.0.1', port=8080, threaded=True), daemon=True).start()

from pyngrok import ngrok, conf
try:   # Kaggle can ship an old ngrok binary that rejects 'domain'; force a fresh download of the current one
    _p = conf.get_default().ngrok_path
    if os.path.exists(_p):
        os.remove(_p)
    ngrok.install_ngrok()
except Exception as e:
    print('ngrok binary refresh skipped:', str(e)[:120])
ngrok.set_auth_token(NGROK)
def open_tunnel():
    for t in ngrok.get_tunnels():
        try: ngrok.disconnect(t.public_url)
        except Exception: pass
    if DOMAIN:
        for kw in ({'domain': DOMAIN}, {'hostname': DOMAIN}):
            try:
                return ngrok.connect(8080, **kw).public_url
            except Exception as e:
                print(f'ngrok with {list(kw)[0]}={DOMAIN} failed: {str(e)[:120]}')
        print('Falling back to a random ngrok URL: copy it into KAGGLE_OLLAMA_URL for this run.')
    return ngrok.connect(8080).public_url
PUBLIC = open_tunnel()
print('PUBLIC URL:', PUBLIC)
if not DOMAIN:
    print('Tip: add an NGROK_DOMAIN secret so this URL stays the same on every run.')


In [ ]:
import base64, io
from PIL import Image, ImageDraw

def smoke():
    im = Image.new('RGB', (320, 200), 'white'); ImageDraw.Draw(im).rectangle((40, 40, 280, 160), fill='red')
    b = io.BytesIO(); im.save(b, 'JPEG')
    r = rq.post(OLLAMA + '/api/chat', timeout=900, json={'model': MODEL, 'stream': False, 'keep_alive': '24h', 'messages': [
        {'role': 'user', 'content': 'What colour is the big rectangle? One word.', 'images': [base64.b64encode(b.getvalue()).decode()]}]})
    r.raise_for_status()
    return r.json()['message']['content']

print('vision test:', smoke())
ps = subprocess.run(['ollama', 'ps'], capture_output=True, text=True).stdout
print(ps)
if 'GPU' not in ps:
    print('WARNING: the model is NOT on the GPU, so replies will be very slow. Set Accelerator to GPU T4 x2 and Run All again.')
print('server is live ->', PUBLIC)

# watchdog: restart Ollama / the tunnel if either dies, and keep the session alive
fails, beat = 0, 0
while True:
    time.sleep(30); beat += 1
    try:
        rq.get(OLLAMA + '/api/tags', timeout=5).raise_for_status(); fails = 0
    except Exception:
        fails += 1
        if fails >= 2:
            print('ollama stopped answering, restarting...')
            try:
                start_ollama(); print('vision test:', smoke()); fails = 0
            except Exception as e:
                print('restart failed:', str(e)[:300])
    try:
        if not ngrok.get_tunnels():
            PUBLIC = open_tunnel(); print('tunnel was down, reopened:', PUBLIC)
    except Exception as e:
        print('tunnel check failed:', str(e)[:200])
    if beat % 20 == 0:
        print('alive', time.strftime('%H:%M:%S'), PUBLIC)
